In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from math import sqrt, dist
import numpy as np
import numba as nb

# =========================
# 1. 场景构造与几何工具 (修改数据部分)
# =========================
def create_complex_steiner_scene():
    """
    三个场景可供选择，取消注释对应场景即可
    """
    
    # ========================================
    # 场景1: 仿生微型机器人 - 室内迷宫探测任务
    # ========================================

    # # 坐标单位：分米（dm），适配微尺度环境
#     start_end_coord = (0.0, 0.0)  # 入口Base
    
#     # 所有点位（包括采样点、中继点、充电点）
#     all_coords = [
#         (0, 0),   # 0: 入口Base
#         (3, 0),   # 1: 采样点A
#         (6, 0),   # 2: 采样点B
#         (9, 0),   # 3: 采样点C
#         (0, 3),   # 4: 采样点D
#         (3, 3),   # 5: 中继点R1
#         (6, 3),   # 6: 采样点E
#         (9, 3),   # 7: 充电点Chg1
#         (0, 6),   # 8: 采样点F
#         (3, 6),   # 9: 采样点G
#         (6, 6),   # 10: 中继点R2
#         (9, 6),   # 11: 采样点H
#         (3, 9),   # 12: 充电点Chg2
#         (6, 9),   # 13: 采样点I
#     ]
    
#     # 必须经过的采样点索引
#     mandatory_indices = [0, 1, 2, 3, 4, 6, 8, 9, 11, 13]
    
#     # 可选经过点（中继点和充电点）
#     optional_indices = [5, 7, 10, 12]
    
#     # 定义可通行边及其代价
#     edge_costs = {
#         (0, 1): 3.2, (1, 2): 3.0, (2, 3): 3.5,
#         (0, 4): 3.0, (1, 5): 4.2, (2, 6): 3.0, (3, 7): 4.5,
#         (4, 5): 3.0, (5, 6): 3.2, (6, 7): 3.0,
#         (4, 8): 3.0, (5, 9): 3.5, (6, 10): 3.0, (7, 11): 3.0,
#         (9, 10): 3.8, (10, 11): 3.2,
#         (9, 12): 5.0, (12, 13): 3.5, (10, 13): 3.0,
#     }
    
#     # 障碍物（迷宫墙壁，通过不可通行的区域隐式表示）
#     obstacles = [
#     {'center': (1.5, 1.5), 'radius': 0.8, 'name': 'Wall-SW'},    # 西南墙壁
#     {'center': (7.5, 1.5), 'radius': 0.8, 'name': 'Wall-SE'},    # 东南墙壁
#     {'center': (4.5, 4.5), 'radius': 1.2, 'name': 'Central-Obs'}, # 中央障碍
#     {'center': (1.5, 7.5), 'radius': 0.8, 'name': 'Wall-NW'},    # 西北墙壁
#     {'center': (7.5, 7.5), 'radius': 0.8, 'name': 'Wall-NE'},    # 东北墙壁
# ]

    
#     # 生成标签
#     point_labels = ['S'] + [f'P{i}' for i in range(1, len(all_coords))] + ['E']
#     all_points = [all_coords[0]] + all_coords[1:] + [all_coords[0]]
#     label_to_coord = {point_labels[i]: all_points[i] for i in range(len(point_labels))}
    
#     # 必经点标签
#     waypoints = [point_labels[i] for i in mandatory_indices if i != 0]
#     optional_points = [point_labels[i] for i in optional_indices]
    '''
    
    # ========================================
    # 场景2: 无人机低空配送 - 园区走廊网络
    # ========================================
    '''
#     # 坐标单位：百米（hm），适配园区尺度
#     start_end_coord = (5.0, 5.0)  # 起降场Depot
    
#     all_coords = [
#         (5, 5),    # 0: 起降场Depot
#         (8, 2),    # 1: 订单屋顶A
#         (14, 2),   # 2: 订单屋顶B
#         (20, 2),   # 3: 订单屋顶C
#         (8, 10),   # 4: 订单屋顶D
#         (14, 10),  # 5: 订单屋顶E
#         (20, 10),  # 6: 订单屋顶F
#         (0, 10),   # 7: 订单屋顶G
#         (4, 5),    # 8: 充电桩C1
#         (12, 6),   # 9: 充电桩C2
#         (18, 6),   # 10: 充电桩C3
#         (6, -4),   # 11: 走廊接入点W1
#         (12, -4),  # 12: 走廊接入点W2
#     ]
    
#     # 必须配送的订单屋顶
#     mandatory_indices = [0, 1, 2, 3, 4, 5, 6, 7]
#     optional_indices = [8, 9, 10, 11, 12]
    
#     # 审批走廊（可飞行边）
#     edge_costs = {
#         (0, 8): 9.0, (8, 1): 6.0, (1, 9): 7.0, (9, 2): 6.0,
#         (2, 10): 7.0, (10, 3): 6.0,
#         (8, 7): 7.0, (7, 4): 7.0, (4, 9): 7.0, (9, 5): 7.0,
#         (5, 10): 7.0, (10, 6): 7.0,
#         (0, 11): 6.5, (11, 12): 6.0, (12, 6): 12.5, (12, 2): 10.0,
#     }
    
#     obstacles = [
#     {'center': (12.5, 6), 'radius': 3.5, 'name': 'Building-Zone'},  # 核心建筑区
#     {'center': (2, 2), 'radius': 1.8, 'name': 'Factory'},           # 工厂禁飞区
#     {'center': (18, 9), 'radius': 2.5, 'name': 'Restricted-Area'},  # 敏感区域
#     {'center': (10, -1), 'radius': 2.0, 'name': 'Ground-Hazard'},   # 地面危险区
# ]

    
#     point_labels = ['S'] + [f'D{i}' for i in range(1, len(all_coords))] + ['E']
#     all_points = [all_coords[0]] + all_coords[1:] + [all_coords[0]]
#     label_to_coord = {point_labels[i]: all_points[i] for i in range(len(point_labels))}
    
#     waypoints = [point_labels[i] for i in mandatory_indices if i != 0]
#     optional_points = [point_labels[i] for i in optional_indices]

    
    # ========================================
    # 场景3: 全球气候观测站巡检网络
    # ========================================
    
    # 坐标：经纬度投影（单位：千公里）
    start_end_coord = (-10.0, -20.0)  # 日内瓦总部Geneva
    
    all_coords = [
        (-10, -20),  # 0: 日内瓦总部Geneva
        (-8, 12),    # 1: 格陵兰冰芯站Greenland-Ice
        (-15, -5),   # 2: 亚马逊雨林站Amazon-Forest
        (6, 15),     # 3: 斯瓦尔巴北极站Svalbard-Arctic
        (18, -8),    # 4: 西伯利亚冻土站Siberia-Permafrost
        (25, 5),     # 5: 青藏高原站Tibet-Plateau
        (22, -15),   # 6: 澳洲内陆站Australia-Outback
        (-12, -18),  # 7: 南极半岛站Antarctic-Peninsula
        (8, -22),    # 8: 南极东方站Antarctic-Vostok
        (12, -2),    # 9: 马尔代夫海平面站Maldives-Sea
        (-20, 2),    # 10: 加拉帕戈斯海洋站Galapagos-Ocean
        (15, 8),     # 11: 喜马拉雅冰川站Himalaya-Glacier
        (28, -3),    # 12: 日本台风观测站Japan-Typhoon
        (-5, 8),     # 13: 冰岛火山站Iceland-Volcano
        (2, 3),      # 14: 阿尔卑斯山站Alps-Mountain
        (-10, -10),  # 15: 巴西高原站Brazil-Savanna
        (20, -12),   # 16: 印尼雨林站Indonesia-Rainforest
        (-18, 10),   # 17: 加拿大北极站Canada-Arctic
        (10, -5),    # 18: 中东沙漠站Dubai-Desert
        (30, 10),    # 19: 蒙古草原站Mongolia-Steppe
    ]
    
    # 必须巡检的关键气候站
    mandatory_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
    optional_indices = [13, 14, 15, 16, 17, 18, 19]
    
    # 国际航线（考虑航权、加油站、极端天气）
    edge_costs = {
        (0, 14): 2.8, (0, 13): 15.6, (13, 1): 18.4, (13, 17): 22.3,
        (1, 17): 12.7, (17, 10): 28.5, (10, 2): 14.6, (2, 15): 18.2,
        (0, 3): 24.8, (3, 4): 32.5, (4, 19): 18.9, (19, 5): 16.3,
        (5, 11): 8.7, (5, 12): 22.4, (12, 16): 25.6, (16, 6): 28.7,
        (6, 7): 42.3, (7, 8): 38.6,
        (0, 18): 26.4, (18, 9): 32.8, (9, 16): 21.5, (18, 5): 28.3,
        (14, 18): 25.7, (11, 19): 19.4, (4, 3): 35.6,
    }
    
    obstacles = []

#     obstacles = [
#     {'center': (4, -5), 'radius': 5.5, 'name': 'Conflict-Zone'},      # 冲突区域
#     {'center': (-5, -25), 'radius': 4.0, 'name': 'Antarctic-Storm'},  # 南极风暴
#     {'center': (20, -13), 'radius': 6.0, 'name': 'Airspace-Restrict'},# 领空限制
#     {'center': (10, -3), 'radius': 4.5, 'name': 'Piracy-Risk'},       # 海盗风险区
#     {'center': (-25, 5), 'radius': 3.5, 'name': 'Atlantic-Hurricane'}, # 大西洋飓风
# ]
    obstacles = [
         {"center": (-5.0, 5.0), "radius": 6.0, "description": "北大西洋风暴区"},
         {"center": (5.0, 0.0), "radius": 5.0, "description": "撒哈拉禁飞区"},
         {"center": (15.0, -5.0), "radius": 7.0, "description": "印度洋无加油区"},
         {"center": (10.0, -20.0), "radius": 10.0, "description": "南冰洋极地风暴"},
         {"center": (20.0, 0.0), "radius": 6.0, "description": "青藏高原边缘禁飞区"}
    ]

    point_labels = ['S'] + [f'C{i}' for i in range(1, len(all_coords))] + ['E']
    all_points = [all_coords[0]] + all_coords[1:] + [all_coords[0]]
    label_to_coord = {point_labels[i]: all_points[i] for i in range(len(point_labels))}
    
    waypoints = [point_labels[i] for i in mandatory_indices if i != 0]
    optional_points = [point_labels[i] for i in optional_indices]
    
    
    return all_points, point_labels, label_to_coord, waypoints, optional_points, obstacles, edge_costs

def line_circle_intersection(p1, p2, center, radius):
    x1, y1 = p1
    x2, y2 = p2
    cx, cy = center
    dx, dy = x2 - x1, y2 - y1
    a = dx * dx + dy * dy
    if a == 0: return (x1 - cx) ** 2 + (y1 - cy) ** 2 <= radius ** 2
    fx, fy = x1 - cx, y1 - cy
    b = 2 * (fx * dx + fy * dy)
    c = (fx * fx + fy * fy) - radius * radius
    discriminant = b * b - 4 * a * c
    if discriminant < 0: return False
    discriminant = sqrt(discriminant)
    t1 = (-b - discriminant) / (2 * a)
    t2 = (-b + discriminant) / (2 * a)
    return (0 <= t1 <= 1) or (0 <= t2 <= 1)

def calculate_edge_weight(p1, p2, obstacles, edge_costs, label1, label2, label_to_coord, obstacle_penalty=1e6, base_distance_weight=1.0):
    """
    修改后的边权计算函数：
    1. 如果edge_costs中定义了该边，优先使用预定义代价
    2. 否则检查障碍物，如果穿过障碍物返回高惩罚
    3. 如果都不满足，返回欧氏距离
    """
    # 查找预定义边（双向）
    all_labels = list(label_to_coord.keys())
    if label1 in all_labels and label2 in all_labels:
        idx1 = all_labels.index(label1)
        idx2 = all_labels.index(label2)
        
        # 注意：S和E指向同一坐标，需要特殊处理
        if label1 == 'S':
            idx1 = 0
        elif label1 == 'E':
            idx1 = 0
        if label2 == 'S':
            idx2 = 0
        elif label2 == 'E':
            idx2 = 0
            
        if (idx1, idx2) in edge_costs:
            return float(edge_costs[(idx1, idx2)])
        elif (idx2, idx1) in edge_costs:
            return float(edge_costs[(idx2, idx1)])
    
    # 检查障碍物
    for obstacle in obstacles:
        if line_circle_intersection(p1, p2, obstacle['center'], obstacle['radius']):
            return float(obstacle_penalty)
    
    # 返回欧氏距离
    return dist(p1, p2) * float(base_distance_weight)

# =========================
# 2. 可视化工具 (保持不变)
# =========================
def visualize_solution(all_points, point_labels, label_to_coord, obstacles, selected_edges, total_distance):
    fig, ax = plt.subplots(figsize=(12, 10))
    for label, coord in label_to_coord.items():
        x, y = coord
        if label == 'S': ax.plot(x, y, 'go', markersize=15, label='Start', markeredgecolor='black', markeredgewidth=2)
        elif label == 'E': ax.plot(x, y, 'ro', markersize=10, label='End', markeredgecolor='black', markeredgewidth=2, alpha=0.7) 
        elif label.startswith('W') or label.startswith('P') or label.startswith('D') or label.startswith('C'): 
            ax.plot(x, y, 'bs', markersize=12, label='Waypoint' if label in ['W1', 'P1', 'D1', 'C1'] else '', markeredgecolor='black', markeredgewidth=2)
        else: ax.plot(x, y, 'ko', markersize=8, alpha=0.5, label='Optional' if label in ['O1'] else '')
        ax.text(x + 0.3, y + 0.3, label, fontsize=9, fontweight='bold')
    for obstacle in obstacles:
        ax.add_patch(patches.Circle(obstacle['center'], obstacle['radius'], edgecolor='red', facecolor='red', alpha=0.3, linewidth=2))
    for u, v in selected_edges:
        p1, p2 = label_to_coord[u], label_to_coord[v]
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], linewidth=2, alpha=0.8)
        mid_x, mid_y = (p1[0] + p2[0]) / 2, (p1[1] + p2[1]) / 2
        ax.arrow(mid_x, mid_y, (p2[0]-p1[0])*0.1, (p2[1]-p1[1])*0.1, head_width=0.5, head_length=0.8, alpha=0.8)
    
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal', 'box')
    ax.set_title(f'Steiner Path Solution (Ising SA -1/+1 Mode)\\nTotal Distance: {total_distance:.2f}', fontsize=14)
    ax.legend(loc='best')
    return fig, ax

@nb.njit
def _numba_sa_core(N_vars, num_reads, t_max, decay_rate, h_isi, J_isi, num_sweeps):
    """
    内部纯数值计算核心：
    - 优化1：降温操作 (np.exp) 提取到 sweep 层。
    - 优化2：利用 CPU 空间局部性进行顺序遍历，替代随机采样。
    """
    # 初始化全局最优记录
    global_best_state = np.zeros(N_vars, dtype=np.int8)
    global_best_energy = np.inf
    global_best_history = np.zeros(num_sweeps, dtype=np.float64)

    for read in range(num_reads):
        # 随机初始化状态 [-1, 1]
        state = np.ones(N_vars, dtype=np.int8)
        for i in range(N_vars):
            if np.random.random() < 0.5:
                state[i] = -1
        
        # 计算初始能量
        current_energy = 0.0
        for i in range(N_vars):
            current_energy += h_isi[i] * state[i]
            for j in range(N_vars):
                current_energy += 0.5 * J_isi[i, j] * state[i] * state[j]
        
        local_best_energy = current_energy
        local_best_state = state.copy()
        current_history = np.zeros(num_sweeps, dtype=np.float64)
        
        # ---------------- 核心优化区域 ----------------
        for sweep in range(num_sweeps):
            # 【优化1】指数降温提取到外层：每个 sweep 仅计算 1 次，而非 N_vars 次
            temp = t_max * np.exp(decay_rate * (sweep / num_sweeps))
            
            # 【优化2】顺序遍历：摒弃 randint，按内存连续地址顺序访问，极大提升缓存命中率
            for idx in range(N_vars):
                s_old = state[idx]
                
                # 计算局部点积用于能量差计算
                dot_J_state = 0.0
                for j in range(N_vars):
                    dot_J_state += J_isi[idx, j] * state[j]
                    
                delta_E = -2.0 * s_old * (h_isi[idx] + dot_J_state)
                
                # Metropolis 接受准则
                if delta_E < 0 or np.random.random() < np.exp(-delta_E / temp):
                    state[idx] = -s_old
                    current_energy += delta_E
                    # 随时记录局部最优
                    if current_energy < local_best_energy:
                        local_best_energy = current_energy
                        local_best_state = state.copy()
            
            # 记录本 sweep 结束后的最优能量
            current_history[sweep] = local_best_energy
        # ----------------------------------------------

        # 更新全局最优
        if local_best_energy < global_best_energy:
            global_best_energy = local_best_energy
            global_best_state = local_best_state.copy()
            global_best_history = current_history.copy()

    return global_best_state, global_best_energy, global_best_history


def custom_simulated_annealing(Q, num_reads=32, num_sweeps=3000, t_max=10.0, t_min=0.01):
    """
    外部接口保持完全不变。
    """
    variables = sorted(list(set(k for pair in Q.keys() for k in pair)))
    N_vars = len(variables)
    var_to_idx = {v: i for i, v in enumerate(variables)}
    
    # 构造 NumPy 矩阵
    h_isi = np.zeros(N_vars, dtype=np.float64)
    J_isi = np.zeros((N_vars, N_vars), dtype=np.float64)
    
    for (u, v), w in Q.items():
        u_idx, v_idx = var_to_idx[u], var_to_idx[v]
        if u_idx == v_idx:
            h_isi[u_idx] += w / 2.0
        else:
            val = w / 4.0
            J_isi[u_idx, v_idx] += val
            J_isi[v_idx, u_idx] += val
            h_isi[u_idx] += val
            h_isi[v_idx] += val

    decay_rate = np.log(t_min / t_max)

    # 移除 total_steps，直接传入 num_sweeps
    global_best_state, global_best_energy, global_best_history = _numba_sa_core(
        N_vars, num_reads, t_max, decay_rate, h_isi, J_isi, num_sweeps
    )
    
    final_sample = {variables[i]: int(global_best_state[i]) for i in range(N_vars)}
    return final_sample, global_best_energy, list(global_best_history)

# =========================
# 4. 显式 QUBO 构建与求解 (透传 history)
# =========================
def solve_steiner_path_explicit(U, u_star, distances, **kwargs):
    print(f"开始构建 QUBO...")
    N = len(U)
    node_to_idx = {node: i for i, node in enumerate(U)}
    idx_to_node = {i: node for i, node in enumerate(U)}
    E_var = lambda u, v: f"e_{u}_{v}"
    X_var = lambda u, v: f"x_{u}_{v}"
    Q = defaultdict(float)

    # 1. 目标函数: 距离
    for u_idx in range(N):
        for v_idx in range(N):
            if u_idx == v_idx: continue
            dist_val = distances.get((idx_to_node[u_idx], idx_to_node[v_idx]), 0.0)
            Q[(E_var(u_idx, v_idx), E_var(u_idx, v_idx))] += kwargs.get('w_obj', 1.0) * dist_val

    # 2. 流约束
    targets = {node_to_idx['S']: (1, 0), node_to_idx['E']: (0, 1)}
    for wp in u_star:
        if wp not in ['S', 'E']: targets[node_to_idx[wp]] = (1, 1)
            
    for idx, (t_out, t_in) in targets.items():
        w = kwargs.get('w_c1', 5.0) if idx in [node_to_idx['S'], node_to_idx['E']] else kwargs.get('w_c2', 10.0)
        out_vars = [E_var(idx, v) for v in range(N) if idx != v]
        for v in out_vars: Q[(v, v)] += w * (1 - 2 * t_out)
        for i in range(len(out_vars)):
            for j in range(i + 1, len(out_vars)): Q[tuple(sorted((out_vars[i], out_vars[j])))] += 2.0 * w
        in_vars = [E_var(u, idx) for u in range(N) if idx != u]
        for v in in_vars: Q[(v, v)] += w * (1 - 2 * t_in)
        for i in range(len(in_vars)):
            for j in range(i + 1, len(in_vars)): Q[tuple(sorted((in_vars[i], in_vars[j])))] += 2.0 * w

    # 3-5. 其它约束
    optional_indices = [i for i in range(N) if i not in targets]
    for idx in optional_indices:
        w = kwargs.get('w_c3', 15.0)
        out_v = [E_var(idx, v) for v in range(N) if idx != v]
        in_v = [E_var(u, idx) for u in range(N) if idx != u]
        for v in out_v + in_v: Q[(v, v)] += w
        for i in range(len(out_v)):
            for j in range(i + 1, len(out_v)): Q[tuple(sorted((out_v[i], out_v[j])))] += 3.0 * w
        for i in range(len(in_v)):
            for j in range(i + 1, len(in_v)): Q[tuple(sorted((in_v[i], in_v[j])))] += 3.0 * w
        for u_v in out_v:
            for v_v in in_v: Q[tuple(sorted((u_v, v_v)))] -= 2.0 * w

    for i in range(N):
        for j in range(i + 1, N): Q[tuple(sorted((E_var(i, j), E_var(j, i))))] += kwargs.get('w_bi', 10.0)

    w4 = kwargs.get('w_c4', 10.0)
    for i in range(N - 1):
        for j in range(i + 1, N):
            x, e_uv, e_vu = X_var(i, j), E_var(i, j), E_var(j, i)
            Q[(e_uv, e_uv)] += w4; Q[tuple(sorted((e_uv, x)))] -= w4; Q[tuple(sorted((e_vu, x)))] += w4
            
    for i in range(N - 2):
        for j in range(i + 1, N - 1):
            for k in range(j + 1, N):
                x_ij, x_jk, x_ik = X_var(i, j), X_var(j, k), X_var(i, k)
                Q[(x_ik, x_ik)] += w4; Q[tuple(sorted((x_ij, x_jk)))] += w4
                Q[tuple(sorted((x_ij, x_ik)))] -= w4; Q[tuple(sorted((x_jk, x_ik)))] -= w4

    # 求解，接收 history
    best_sample, best_energy, history = custom_simulated_annealing(
        Q, 
        num_reads=kwargs.get('num_reads', 32), 
        num_sweeps=kwargs.get('num_sweeps', 3000)
    )
    
    # 解析
    selected_edges, total_distance = [], 0.0
    for u_idx in range(N):
        for v_idx in range(N):
            if u_idx == v_idx: continue
            var = E_var(u_idx, v_idx)
            if best_sample.get(var, -1) == 1: 
                u, v = idx_to_node[u_idx], idx_to_node[v_idx]
                selected_edges.append((u, v))
                total_distance += distances.get((u, v), 0.0)
    return selected_edges, total_distance, best_energy, history

# =========================
# 5. 主程序 (修改以支持edge_costs)
# =========================
def main():
    all_points, point_labels, label_to_coord, waypoints, optional_points, obstacles, edge_costs = create_complex_steiner_scene()
    
    U = point_labels
    u_star = ['S'] + waypoints + ['E']
    distances = {}
    for u in U:
        for v in U:
            if u == v: continue
            distances[(u, v)] = calculate_edge_weight(
                label_to_coord[u], label_to_coord[v], obstacles, 
                edge_costs, u, v, label_to_coord
            )

    # 增加 num_sweeps 以适应更复杂的问题规模，并解包 history
    edges, dist_val, energy, history = solve_steiner_path_explicit(U, u_star, distances, num_reads=32, num_sweeps=3000)
    
    print(f"\n求解结果 (Ising Mode):\n总距离: {dist_val:.2f}\nIsing能量: {energy:.2f}")
    if edges:
        path_dict, curr, path_str, visited = {u:v for u, v in edges}, 'S', ['S'], {'S'}
        while curr != 'E' and curr in path_dict:
            curr = path_dict[curr]
            if curr in visited: break
            visited.add(curr); path_str.append(curr)
        print("路径顺序:", " -> ".join(path_str))
    
    # === 新增：能量收敛曲线可视化 ===
    plt.figure(figsize=(10, 5))
    plt.plot(history, label='Best Minimum Energy', color='blue', linewidth=1.5)
    plt.title('Simulated Annealing Convergence (Best Run)') 
    plt.xlabel('Sweeps')
    plt.ylabel('Minimum Energy')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.show()
    # ==============================

                       
    visualize_solution(all_points, point_labels, label_to_coord, obstacles, edges, dist_val)
    plt.show()

if __name__ == "__main__":
    main()



TypeError: unhashable type: 'dict'